# DAT-02 — Kiểm tra pilot JMA local

Development-only; không fit model, không gọi Docker/MinIO/API và không thay official dataset lifecycle. Feature do Java/Spark MLD-04 tạo. Chạy Restart Kernel → Run All. Dùng `make inspect-pilot PILOT_BUNDLE=...` để nghiệm thu tự động; `make jupyter-pilot PILOT_BUNDLE=...` để mở tương tác.

In [ ]:
import os
import sys
from pathlib import Path
root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
assert (root / 'notebooks/pilot_bundle.py').is_file(), 'Mở notebook từ root repo hoặc notebooks/'
sys.path.insert(0, str(root / 'notebooks'))
from pilot_bundle import load_bundle
bundle_path = os.environ.get('PILOT_BUNDLE', '')
assert bundle_path, 'Đặt PILOT_BUNDLE trỏ tới đúng pilot_id directory; không tự chọn latest'
bundle = load_bundle(Path(bundle_path).resolve())
print(bundle['summary'])

## Schema, số lượng và vector

Loader đã kiểm checksum/schema/finite/grain/self/lineage/coverage trước khi trả dữ liệu. Magnitude là metadata, không nằm trong vector 4-D. Không tính lại feature hoặc fit scaler trong notebook.

In [ ]:
manifest = bundle['manifest']
print('Feature schema:', bundle['features'].schema.remove_metadata())
print('Counts:', {'features': bundle['features'].num_rows, 'mainshocks': bundle['mainshocks'].num_rows})
print('Vector:', manifest['vector_columns'])
print('Scaling:', manifest['scaling_config_json'])
assert manifest['research_ready'] is False
assert manifest['source_dataset_status'] == 'BUILDING'

## Coverage và bàn giao

PARTIAL không phải lỗi exporter: slice 14 ngày không bao phủ toàn bộ window model. Không cắt window để đổi nhãn thành COMPLETE. Window self-only được giữ nguyên; bộ nhỏ không chứng minh chất lượng clustering.

In [ ]:
for window in bundle['coverage']['windows']:
    print({'mainshock': window['mainshock_event_id'], 'candidates': window['candidate_count'],
           'status': window['status'], 'missing_ranges': window['missing_ranges']})
assert bundle['summary']['status'] == 'PILOT_INSPECTION_PASSED'
print('DAT-02 INSPECTION PASSED — DEVELOPMENT_ONLY; EXP-01/EXP-02 tiếp tục harness/fit, MLD-05 export official, MLI-03 import official')